In [1]:
import qlib
import os
from qlib.contrib.model import LGBModel
from qlib.data.dataset import DatasetH
from qlib.workflow import R
import pandas as pd

ModuleNotFoundError. CatBoostModel are skipped. (optional: maybe installing CatBoostModel can fix it.)


In [2]:
qlib.init(provider_uri="~/.qlib/qlib_data/amz_data", region="cn")
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

[4724:MainThread](2026-08-15 14:04:47,744) INFO - qlib.Initialization - [config.py:473] - default_conf: client.
[4724:MainThread](2026-08-15 14:04:47,749) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[4724:MainThread](2026-08-15 14:04:47,751) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Users/Joe/.qlib/qlib_data/amz_data')}


In [24]:
handler_config = {
        "start_time": "2014-01-02",
        "end_time": "2026-08-14",
        "fit_start_time": "2014-01-01",
        "fit_end_time": "2020-01-01",
        "instruments": ["SH518880"],
        "learn_processors": [{"class": "DropnaLabel"}],
    }

In [25]:
dataset = DatasetH(
        handler={
            "class": "Alpha158",
            "module_path": "qlib.contrib.data.handler",
            "kwargs": handler_config,
        },
        segments={
            "train": ("2014-01-02", "2020-01-01"),
            "valid": ("2020-01-02", "2021-01-01"),
            "test": ("2021-01-02", "2026-08-14"),
        }
    )

[4724:MainThread](2026-08-15 14:28:26,195) INFO - qlib.timer - [log.py:127] - Time cost: 0.060s | Loading data Done
[4724:MainThread](2026-08-15 14:28:26,197) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | DropnaLabel Done
[4724:MainThread](2026-08-15 14:28:26,197) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done
[4724:MainThread](2026-08-15 14:28:26,197) INFO - qlib.timer - [log.py:127] - Time cost: 0.062s | Init data Done


In [26]:
model = LGBModel(
        eval_metric="rmse",
        colsample_bytree=0.8879,
        learning_rate=0.2,
        subsample=0.8789,
        lambda_l1=205.6999,
        lambda_l2=580.9768,
        max_depth=8,
        num_leaves=210,
        num_threads=20
    )

In [27]:
with R.start(experiment_name="test"):
    model.fit(dataset)
    recorder = R.get_recorder()
    print(f"实验ID: {recorder.id}")

[4724:MainThread](2026-08-15 14:28:29,810) INFO - qlib.workflow - [exp.py:258] - Experiment 800168855949788052 starts running ...
[4724:MainThread](2026-08-15 14:28:29,834) INFO - qlib.workflow - [recorder.py:345] - Recorder bc465320016948259fb1be55e4734e23 starts running under Experiment 800168855949788052 ...


Training until validation scores don't improve for 50 rounds
[20]	train's l2: 5.77143e-05	valid's l2: 0.000142117
[40]	train's l2: 5.77143e-05	valid's l2: 0.000142117
Early stopping, best iteration is:
[1]	train's l2: 5.77143e-05	valid's l2: 0.000142117
实验ID: bc465320016948259fb1be55e4734e23


[4724:MainThread](2026-08-15 14:28:30,480) INFO - qlib.timer - [log.py:127] - Time cost: 0.245s | waiting `async_log` Done


In [28]:
predictions = model.predict(dataset)
predictions

datetime    instrument
2021-01-04  SH518880      0.000264
2021-01-05  SH518880      0.000264
2021-01-06  SH518880      0.000264
2021-01-07  SH518880      0.000264
2021-01-08  SH518880      0.000264
                            ...   
2026-08-10  SH518880      0.000264
2026-08-11  SH518880      0.000264
2026-08-12  SH518880      0.000264
2026-08-13  SH518880      0.000264
2026-08-14  SH518880      0.000264
Length: 1361, dtype: float64

In [14]:
from qlib.data.dataset.handler import DataHandlerLP

df_train, df_valid = dataset.prepare(
    ["train", "valid"],
    col_set=["feature", "label"],
    data_key=DataHandlerLP.DK_L,
)
print(df_train["label"].describe())
print(df_valid["label"].describe())

       LABEL0
count     0.0
mean      NaN
std       NaN
min       NaN
25%       NaN
50%       NaN
75%       NaN
max       NaN
       LABEL0
count     0.0
mean      NaN
std       NaN
min       NaN
25%       NaN
50%       NaN
75%       NaN
max       NaN


In [23]:
from qlib.data import D
close = D.features(instruments=["SH518880"], fields=["$close"], start_time="2015-01-01", end_time="2016-01-10", freq="day")
print(close.isna().mean(), close.eq(0).mean())

$close    0.0
dtype: float64 $close    0.0
dtype: float64
